# Simulación de portafolio con Cholesky
María Fernanda Gómez Flores

Simulación de AAPL, MSFT y NVDA a un año con 10,000 escenarios.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

### Precios y rendimientos
Descargamos los precios de las acciones y sacamos los rendimientos logarítmicos diarios.

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA"]
prices = yf.download(tickers=tickers, start="2023-11-01", end="2026-11-01")["Close"]
prices = prices[tickers].dropna()
rets = np.log(prices / prices.shift(1)).dropna()
rets

### Correlación
Calculamos la correlación entre los rendimientos de las tres acciones.

In [ ]:
rets.corr()

### Covarianza
Sacamos la matriz de covarianzas de los rendimientos diarios.

In [ ]:
Sigma = rets.cov()
Sigma

### Cholesky
Descomponemos la matriz de covarianzas para poder simular los rendimientos tomando en cuenta la relación entre las acciones.

In [ ]:
L = np.linalg.cholesky(Sigma)
L

### Rendimiento promedio
Calculamos el rendimiento promedio diario y lo pasamos a anual usando 252 días.

In [ ]:
mu = rets.mean().values * 252
mu

### Simulación
Generamos 10,000 números aleatorios normales para cada acción.

In [ ]:
M = 10000
rng = np.random.default_rng(42)
epsilon = rng.standard_normal((len(tickers), M))
epsilon

### Rendimientos correlacionados
Multiplicamos los números aleatorios por la matriz de Cholesky para obtener los movimientos correlacionados.

In [ ]:
Z = L @ epsilon
Z.T

### Precios futuros
Simulamos los precios de cada acción dentro de un año. Aquí ya estamos usando la volatilidad que viene de la matriz de covarianzas.

In [ ]:
T = 1
n_dias = 252
S0 = prices.iloc[-1].values
mu = rets.mean().values * n_dias
vol = rets.std().values * np.sqrt(n_dias)

ST = S0 * np.exp(
    (mu - 0.5 * vol**2) * T
    + np.sqrt(n_dias * T) * Z.T
)
ST = pd.DataFrame(ST, columns=tickers)
ST.head()

### Pesos del portafolio
Vamos a usar tres formas distintas de repartir los $100,000 entre AAPL, MSFT y NVDA. En el primero invertimos todo en AAPL, en el segundo repartimos casi igual y en el tercero le damos más peso a NVDA.

In [ ]:
V0 = 100000

pesos = {
    "Escenario 1": np.array([1.00, 0.00, 0.00]),
    "Escenario 2": np.array([0.33, 0.33, 0.34]),
    "Escenario 3": np.array([0.20, 0.20, 0.60])
}

pd.DataFrame(pesos, index=tickers).T

### Valor futuro de los tres portafolios
Con los mismos precios simulados calculamos cuántas acciones tendríamos en cada escenario y cuánto valdría el portafolio al final del año.

In [ ]:
resultados = {}

for nombre, w in pesos.items():
    acciones = V0 * w / S0
    VT = ST.to_numpy() @ acciones
    PnL = VT - V0

    resultados[nombre] = {"VT": VT, "PnL": PnL}

pd.DataFrame({nombre: datos["VT"] for nombre, datos in resultados.items()}).head()

### Gráfica del valor futuro
Comparamos cómo se distribuye el valor de los tres portafolios antes de restarle el capital inicial. La línea marca los $100,000 que invertimos.

In [ ]:
plt.figure(figsize=(10, 6))

for nombre, datos in resultados.items():
    plt.hist(datos["VT"], bins=50, alpha=0.4, label=nombre)

plt.axvline(V0, color="black", linestyle="--", label="Capital inicial")
plt.title("Distribución del valor futuro")
plt.xlabel("Valor futuro del portafolio ($)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

### Ganancias y pérdidas
Ahora a cada valor futuro le restamos los $100,000 iniciales.

$$P\&L = V_T - V_0$$

Los valores negativos son pérdidas y los positivos son ganancias.

In [ ]:
pd.DataFrame({nombre: datos["PnL"] for nombre, datos in resultados.items()}).head()

### Gráfica de ganancias y pérdidas
Graficamos los tres escenarios después de restar el capital inicial. El cero significa que recuperamos exactamente lo que invertimos.

In [ ]:
plt.figure(figsize=(10, 6))

for nombre, datos in resultados.items():
    plt.hist(datos["PnL"], bins=50, alpha=0.4, label=nombre)

plt.axvline(0, color="black", linestyle="--", label="Sin ganancia ni pérdida")
plt.title("Distribución de ganancias y pérdidas")
plt.xlabel("Ganancia / Pérdida ($)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

### Comparación de escenarios
Calculamos la ganancia promedio, la volatilidad del rendimiento simulado y la probabilidad de terminar con pérdidas para comparar qué tanto cambia el resultado con los pesos.

In [ ]:
resumen = []

for nombre, datos in resultados.items():
    pnl = datos["PnL"]
    resumen.append({
        "Escenario": nombre,
        "Valor futuro promedio ($)": datos["VT"].mean(),
        "Ganancia promedio ($)": pnl.mean(),
        "Volatilidad del rendimiento (%)": np.std(pnl / V0, ddof=1) * 100,
        "Probabilidad de pérdida (%)": np.mean(pnl < 0) * 100
    })

comparacion = pd.DataFrame(resumen).set_index("Escenario")
comparacion.round(2)